# 08 — Hard-Negative-Training mit Kreuzvalidierung

Wiederholung des Hard-Negative-Versuchs aus Notebook 03 mit
fuenffacher, subjektunabhaengiger Kreuzvalidierung.

**Warum diese Wiederholung noetig ist**

Das Ergebnis hat sich zwischen zwei Durchlaeufen umgekehrt:

| Durchlauf | FT ohne | FT mit | Bewertung |
|---|---|---|---|
| vor Label-Korrektur | Se 0,723 | Se 0,638 | schadet |
| nach Label-Korrektur | Se 0,261 | Se 0,696 | hilft deutlich |

Ein Vorzeichenwechsel dieser Groessenordnung bei einer einzelnen
Aufteilung ist ein Hinweis auf Zufallsschwankung. Zudem sank im
zweiten Durchlauf die Spezifitaet (0,967 auf 0,950) bei steigender
Zahl von Fehlalarmen (87 auf 130) -- Sensitivitaet allein genuegt
hier also nicht zur Beurteilung.

Geprueft wird daher ueber alle Falten, ob die Gewichtung
impactstarker ADL-Fenster einen belastbaren Vorteil bringt.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"

def hole(name, ziel):
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        vorhanden = "\n    ".join(p.name for p in sorted(Path(BASE).iterdir()))
        raise FileNotFoundError(
            f"'{name}' nicht gefunden unter {BASE}\n"
            f"  Vorhandene Ordner:\n    {vorhanden}")
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

from tensorflow import keras
if not Path(MODEL).exists():
    raise FileNotFoundError(f"{MODEL} fehlt - bitte 01 ausfuehren.")
basis = keras.models.load_model(MODEL, compile=False)
basis.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Basismodell: {basis.count_params():,} Parameter")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Basismodell: 61,601 Parameter


In [2]:
N_FOLDS     = 5
POSITION    = "ANKLE"
HARD_WEIGHT = 3.0     # Gewichtsfaktor fuer impactstarke ADL-Fenster
PEAK_THR    = 1.8     # Schwelle in g

run = fl.new_run("08_hardneg_cv")

uma = fl.load_umafall(
    hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
         "/content/UMAFall"), position=POSITION)

# Fensterung mit korrekter Beschriftung: innerhalb einer
# Sturzaufzeichnung gilt nur das Fenster als Sturz, das den
# Aufprall enthaelt.
X_raw, y, g, akt = fl.build_windows(uma, mode="sliding")
peaks = fl.acc_magnitude(X_raw).max(axis=1)   # vor der Normalisierung
X = fl.znorm(X_raw)

hart = (y == 0) & (peaks > PEAK_THR)
print(f"\n{len(y)} Fenster, {int(y.sum())} Stuerze")
print(f"Hard Negatives: {int(hart.sum())} von {int((y==0).sum())} "
      f"ADL-Fenstern ({100*hart.sum()/max((y==0).sum(),1):.1f}%)")

[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/08_hardneg_cv_2026-08-26_1321
[ok] UMAFall Fall Detection/UMAFall_Dataset_corrected bereits lokal
[UMAFall ANKLE] 643 Trials (157 Sturz, 486 ADL), 19 Probanden

8369 Fenster, 299 Stuerze
Hard Negatives: 2160 von 8070 ADL-Fenstern (26.8%)


### Welche Aktivitaeten liefern die Hard Negatives?

Zur Kontrolle, ob die Auswahl tatsaechlich die erwarteten
Bewegungen mit Bodenkontakt trifft.

In [3]:
print(f"{'Aktivitaet':<26} {'ADL-Fenster':>12} {'davon hart':>11} {'Anteil':>8}")
print("-"*60)
verteilung = {}
for a in sorted(np.unique(akt)):
    m = (akt == a) & (y == 0)
    if m.sum() == 0:
        continue
    h = int((hart & m).sum())
    print(f"{a:<26} {int(m.sum()):>12d} {h:>11d} "
          f"{100*h/m.sum():>7.1f}%")
    verteilung[a] = {"adl_windows": int(m.sum()), "hard": h,
                     "share": round(float(h/m.sum()), 4)}

Aktivitaet                  ADL-Fenster  davon hart   Anteil
------------------------------------------------------------
Aplausing                           548           0     0.0%
Bending                             648           3     0.5%
GoDownstairs                        259         133    51.4%
GoUpstairs                          255         134    52.5%
HandsUp                             567          20     3.5%
Hopping                             584         493    84.4%
Jogging                             259         223    86.1%
LyingDown_OnABed                    624         178    28.5%
MakingACall                         610           3     0.5%
OpeningDoor                         563         124    22.0%
Sitting_GettingUpOnAChair           699          52     7.4%
Walking                             713         604    84.7%
backwardFall                        630          76    12.1%
forwardFall                         591          72    12.2%
lateralFall             

## Kreuzvalidierung

In [4]:
splits = fl.group_kfold_splits(g, y, n_splits=N_FOLDS)

# Gewichte: Klassenbalance mal Hard-Negative-Faktor
w_basis = np.ones(len(y), np.float32)
cw = fl.class_weights(y)
w_basis[y == 1] *= cw[1]
w_basis[y == 0] *= cw[0]
w_hart = w_basis.copy()
w_hart[hart] *= HARD_WEIGHT

falten = {"basis": [], "ohne_hard": [], "mit_hard": []}

for i, (tr, te) in enumerate(splits, 1):
    n_f = int(y[te].sum())
    print(f"\nFalte {i}/{N_FOLDS}: Test {len(te)} Fenster, "
          f"{n_f} Stuerze, {len(np.unique(g[te]))} Probanden")
    if n_f == 0:
        print("  keine Stuerze im Testsatz - uebersprungen")
        continue

    falten["basis"].append(
        fl.evaluate(basis, X[te], y[te], "  Basismodell"))

    m_ohne = fl.finetune(basis, X[tr], y[tr],
                         sample_weight=w_basis[tr])
    falten["ohne_hard"].append(
        fl.evaluate(m_ohne, X[te], y[te], "  FT ohne Hard Neg."))

    m_mit = fl.finetune(basis, X[tr], y[tr],
                        sample_weight=w_hart[tr])
    falten["mit_hard"].append(
        fl.evaluate(m_mit, X[te], y[te], "  FT mit Hard Neg."))


Falte 1/5: Test 1333 Fenster, 44 Stuerze, 3 Probanden
    Basismodell               Se=0.159  Sp=0.967  Acc=0.940  FP=   43  FN=  37
    FT ohne Hard Neg.         Se=0.886  Sp=0.943  Acc=0.941  FP=   73  FN=   5


    FT mit Hard Neg.          Se=0.818  Sp=0.961  Acc=0.956  FP=   50  FN=   8

Falte 2/5: Test 2466 Fenster, 50 Stuerze, 4 Probanden
    Basismodell               Se=0.080  Sp=0.987  Acc=0.969  FP=   31  FN=  46
    FT ohne Hard Neg.         Se=0.680  Sp=0.936  Acc=0.931  FP=  154  FN=  16
    FT mit Hard Neg.          Se=0.580  Sp=0.957  Acc=0.949  FP=  104  FN=  21

Falte 3/5: Test 2492 Fenster, 119 Stuerze, 6 Probanden
    Basismodell               Se=0.143  Sp=0.973  Acc=0.933  FP=   65  FN= 102
    FT ohne Hard Neg.         Se=0.689  Sp=0.903  Acc=0.893  FP=  229  FN=  37
    FT mit Hard Neg.          Se=0.597  Sp=0.933  Acc=0.917  FP=  159  FN=  48

Falte 4/5: Test 912 Fenster, 36 Stuerze, 3 Probanden
    Basismodell               Se=0.056  Sp=0.974  Acc=0.938  FP=   23  FN=  34
    FT ohne Hard Neg.         Se=0.778  Sp=0.928  Acc=0.922  FP=   63  FN=   8


    FT mit Hard Neg.          Se=0.667  Sp=0.950  Acc=0.939  FP=   44  FN=  12

Falte 5/5: Test 1166 Fenster, 50 Stuerze, 3 Probanden
    Basismodell               Se=0.140  Sp=0.979  Acc=0.943  FP=   23  FN=  43
    FT ohne Hard Neg.         Se=0.880  Sp=0.941  Acc=0.938  FP=   66  FN=   6
    FT mit Hard Neg.          Se=0.840  Sp=0.953  Acc=0.948  FP=   53  FN=   8


## Zusammengefasstes Ergebnis

In [5]:
gepoolt = {k: fl.pool_folds(v) for k, v in falten.items() if v}

print(f"Ueber {N_FOLDS} Falten zusammengefasst:\n")
namen = {"basis": "Basismodell",
         "ohne_hard": "FT ohne Hard Neg.",
         "mit_hard": "FT mit Hard Neg."}
for k, n in namen.items():
    if k in gepoolt:
        fl.print_pooled(n, gepoolt[k])

print(f"\n{'Variante':<20} {'Se je Falte'}")
print("-"*54)
for k, n in namen.items():
    if k in gepoolt:
        print(f"{n:<20} {gepoolt[k]['Se_per_fold']}")

Ueber 5 Falten zusammengefasst:

  Basismodell            Se=0.124 [0.091, 0.166]  Sp=0.977  (TP=37, FN=262, FP=185)
  FT ohne Hard Neg.      Se=0.759 [0.708, 0.804]  Sp=0.927  (TP=227, FN=72, FP=585)
  FT mit Hard Neg.       Se=0.676 [0.621, 0.726]  Sp=0.949  (TP=202, FN=97, FP=410)

Variante             Se je Falte
------------------------------------------------------
Basismodell          [np.float64(0.1591), np.float64(0.08), np.float64(0.1429), np.float64(0.0556), np.float64(0.14)]
FT ohne Hard Neg.    [np.float64(0.8864), np.float64(0.68), np.float64(0.6891), np.float64(0.7778), np.float64(0.88)]
FT mit Hard Neg.     [np.float64(0.8182), np.float64(0.58), np.float64(0.5966), np.float64(0.6667), np.float64(0.84)]


## Gepaarter Vergleich

Bewertet werden Sensitivitaet **und** Spezifitaet: Eine Massnahme,
die mehr Stuerze erkennt, aber ebenso viele zusaetzliche Fehlalarme
erzeugt, ist fuer den Einsatz in der Sohle kein Gewinn.

In [6]:
vergleiche = {}
if falten["ohne_hard"] and falten["mit_hard"]:
    for metric in ["Se", "Sp"]:
        print()
        vergleiche[f"hard_vs_ohne_{metric}"] = fl.paired_comparison(
            falten["ohne_hard"], falten["mit_hard"],
            "ohne Hard Neg.", "mit Hard Neg.", metric=metric)


  Gepaarter Vergleich (Se): mit Hard Neg. minus ohne Hard Neg.
    je Falte: [-0.068, -0.1, -0.093, -0.111, -0.04]
    Mittelwert: -8.2 pp  (95 %-KI [-10.7, -5.7] pp)
    mit Hard Neg. besser in 0 von 5 Falten
    -> Vorteil fuer ohne Hard Neg. statistisch gesichert.

  Gepaarter Vergleich (Sp): mit Hard Neg. minus ohne Hard Neg.
    je Falte: [0.018, 0.021, 0.03, 0.022, 0.012]
    Mittelwert: +2.0 pp  (95 %-KI [+1.5, +2.6] pp)
    mit Hard Neg. besser in 5 von 5 Falten
    -> Vorteil fuer mit Hard Neg. statistisch gesichert.


## Einordnung

In [7]:
vs, vp = (vergleiche.get("hard_vs_ohne_Se"),
          vergleiche.get("hard_vs_ohne_Sp"))
if vs and vp:
    o, m = gepoolt["ohne_hard"], gepoolt["mit_hard"]
    print(f"{'':22} {'ohne':>10} {'mit':>10} {'Differenz':>12}")
    print("-"*58)
    print(f"{'Sensitivitaet':<22} {o['Se']:>10.3f} {m['Se']:>10.3f} "
          f"{vs['mean_delta']*100:>+11.1f}pp")
    print(f"{'Spezifitaet':<22} {o['Sp']:>10.3f} {m['Sp']:>10.3f} "
          f"{vp['mean_delta']*100:>+11.1f}pp")
    print(f"{'Fehlalarme (FP)':<22} {o['FP']:>10d} {m['FP']:>10d} "
          f"{m['FP']-o['FP']:>+12d}")
    print(f"{'Verpasste (FN)':<22} {o['FN']:>10d} {m['FN']:>10d} "
          f"{m['FN']-o['FN']:>+12d}")

    print()
    if vs["significant"] and vs["mean_delta"] > 0 and \
       not (vp["significant"] and vp["mean_delta"] < -0.02):
        print("Die Gewichtung verbessert die Sensitivitaet gesichert,")
        print("ohne die Spezifitaet nennenswert zu beeintraechtigen.")
        print("Der Ansatz ist fuer die Zielhardware zu empfehlen.")
    elif vs["significant"] and vp["significant"] and vp["mean_delta"] < 0:
        print("Die Gewichtung erhoeht die Sensitivitaet, senkt aber")
        print("zugleich gesichert die Spezifitaet. Ob dieser Tausch")
        print("lohnt, haengt davon ab, wie viele Fehlalarme im Alltag")
        print("hinnehmbar sind.")
    else:
        print("Kein gesicherter Effekt. Die gegenlaeufigen Ergebnisse")
        print("frueherer Einzelaufteilungen sind demnach auf")
        print("Zufallsschwankung zurueckzufuehren; der Ansatz wird")
        print("nicht weiterverfolgt.")

fl.save_results(run,
    {"gepoolt": gepoolt, "vergleiche": vergleiche, "falten": falten,
     "config": {"hard_weight": HARD_WEIGHT, "peak_g": PEAK_THR,
                "n_hard": int(hart.sum()), "n_folds": N_FOLDS},
     "hard_negatives_nach_aktivitaet": verteilung},
    config=fl.default_config(),
    notes=("Ersetzt den Hard-Negative-Teil von Notebook 03. "
           "Fuenffache Kreuzvalidierung, gepaarter Vergleich fuer "
           "Sensitivitaet und Spezifitaet."))

                             ohne        mit    Differenz
----------------------------------------------------------
Sensitivitaet               0.759      0.676        -8.2pp
Spezifitaet                 0.927      0.949        +2.0pp
Fehlalarme (FP)               585        410         -175
Verpasste (FN)                 72         97          +25

Kein gesicherter Effekt. Die gegenlaeufigen Ergebnisse
frueherer Einzelaufteilungen sind demnach auf
Zufallsschwankung zurueckzufuehren; der Ansatz wird
nicht weiterverfolgt.
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/08_hardneg_cv_2026-08-26_1321/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/08_hardneg_cv_2026-08-26_1321/results.json')

In [8]:
import re
from pathlib import Path
bad = []
for f in sorted(Path("/content/UMAFall").rglob("*.csv")):
    if not re.search(r"_(?:ADL|FALL)_(.+?)_\d+_\d{4}-", f.stem):
        bad.append(f.stem)
print(f"{len(bad)} von {len(list(Path('/content/UMAFall').rglob('*.csv')))} Dateien")
for b in bad[:15]: print(" ", b)

208 von 746 Dateien
  UMAFall_Subject_02_Fall_backwardFall_1_2016-06-13_20-51-32
  UMAFall_Subject_02_Fall_backwardFall_2_2016-06-13_20-52-16
  UMAFall_Subject_02_Fall_backwardFall_3_2016-06-13_20-52-55
  UMAFall_Subject_02_Fall_backwardFall_4_2016-06-13_20-58-40
  UMAFall_Subject_02_Fall_backwardFall_5_2016-06-13_20-59-19
  UMAFall_Subject_02_Fall_backwardFall_6_2016-06-13_20-59-59
  UMAFall_Subject_02_Fall_forwardFall_1_2016-06-13_20-43-52
  UMAFall_Subject_02_Fall_forwardFall_2_2016-06-13_20-45-53
  UMAFall_Subject_02_Fall_forwardFall_2_2016-06-13_20-48-20
  UMAFall_Subject_02_Fall_forwardFall_3_2016-06-13_20-47-36
  UMAFall_Subject_02_Fall_forwardFall_4_2016-06-13_20-53-39
  UMAFall_Subject_02_Fall_forwardFall_5_2016-06-13_20-54-39
  UMAFall_Subject_02_Fall_forwardFall_6_2016-06-13_20-55-19
  UMAFall_Subject_02_Fall_lateralFall_1_2016-06-13_20-49-17
  UMAFall_Subject_02_Fall_lateralFall_2_2016-06-13_20-50-00


In [9]:
import inspect, fall_lib as fl

quelle = inspect.getsource(fl.load_umafall)
neu = "IGNORECASE" in quelle
print("Geladene Version:", "NEU (mit IGNORECASE)" if neu else "ALT")
print()

# Was liegt auf der Platte?
disk = open("/content/fall_lib.py").read()
print("Datei auf Platte:", "NEU" if "IGNORECASE" in disk else "ALT")
print("Groesse:", len(disk), "Zeichen")

Geladene Version: NEU (mit IGNORECASE)

Datei auf Platte: NEU
Groesse: 46940 Zeichen


In [10]:
import re
from pathlib import Path

dateien = list(Path("/content/UMAFall").rglob("*.csv"))
fehl = [f.stem for f in dateien
        if not re.search(r"_(?:ADL|FALL)_(.+?)_\d+_\d{4}-",
                         f.stem, re.IGNORECASE)]
print(f"Nicht erkannt: {len(fehl)} von {len(dateien)}   (erwartet: 0)")

# Direkt ueber den Loader pruefen
tr = fl.load_umafall("/content/UMAFall", position="ANKLE", verbose=False)
from collections import Counter
akt = Counter(t["activity"] for t in tr)
print(f"unknown-Trials: {akt.get('unknown', 0)}   (erwartet: 0)")
print(f"Aktivitaeten: {sorted(akt)}")

Nicht erkannt: 0 von 746   (erwartet: 0)
unknown-Trials: 0   (erwartet: 0)
Aktivitaeten: ['Aplausing', 'Bending', 'GoDownstairs', 'GoUpstairs', 'HandsUp', 'Hopping', 'Jogging', 'LyingDown_OnABed', 'MakingACall', 'OpeningDoor', 'Sitting_GettingUpOnAChair', 'Walking', 'backwardFall', 'forwardFall', 'lateralFall']


In [11]:
import re, inspect
from pathlib import Path
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

# Eine Sturzdatei herausgreifen
sturz = [f for f in sorted(Path("/content/UMAFall").rglob("*.csv"))
         if "_FALL_" in f.stem.upper()][0]
print("Datei:", sturz.stem)
print("Ordner:", sturz.parent)
print()

muster = r"_(?:ADL|FALL)_(.+?)_\d+_\d{4}-"
print("direkt getestet :", re.search(muster, sturz.stem, re.IGNORECASE))
print()

# Genau die Zeile aus dem geladenen Modul anzeigen
for zeile in inspect.getsource(fl.load_umafall).splitlines():
    if "ma = re.search" in zeile or "ma.group" in zeile or "IGNORECASE" in zeile:
        print("  Modul:", zeile.strip())
print()

# Was macht der Loader mit genau dieser Datei?
t = [x for x in fl.load_umafall("/content/UMAFall", position="ANKLE",
                                verbose=False)
     if x["label"] == 1]
print(f"Sturz-Trials geladen: {len(t)}")
if t:
    print("  activity des ersten:", repr(t[0]["activity"]))

Datei: UMAFall_Subject_02_Fall_backwardFall_1_2016-06-13_20-51-32
Ordner: /content/UMAFall

direkt getestet : <re.Match object; span=(18, 44), match='_Fall_backwardFall_1_2016-'>

  Modul: ms = re.search(r"Subject_(\d+)", f.stem, re.IGNORECASE)
  Modul: # statt "_FALL_". Ohne IGNORECASE bleibt der Aktivitaetsname
  Modul: ma = re.search(r"_(?:ADL|FALL)_(.+?)_\d+_\d{4}-", f.stem,
  Modul: re.IGNORECASE)
  Modul: "activity": ma.group(1) if ma else "unknown",

Sturz-Trials geladen: 157
  activity des ersten: 'backwardFall'


In [12]:
import inspect, fall_lib as fl
print(inspect.getsource(fl.load_umafall))

def load_umafall(root, position="ANKLE", verbose=True):
    """UMAFall fuer eine bestimmte Sensorposition."""
    position = position.upper()
    trials = []
    for f in sorted(Path(root).rglob("*.csv")):
        is_fall = "_FALL_" in f.stem.upper()
        ms = re.search(r"Subject_(\d+)", f.stem, re.IGNORECASE)
        # Die Schreibweise des Klassenmarkers schwankt zwischen den
        # Dateien: 208 der 746 Aufzeichnungen verwenden "_Fall_"
        # statt "_FALL_". Ohne IGNORECASE bleibt der Aktivitaetsname
        # dieser Dateien unerkannt. Die Klassenzuordnung oben ist
        # davon nicht betroffen, da sie ohnehin auf .upper() beruht.
        ma = re.search(r"_(?:ADL|FALL)_(.+?)_\d+_\d{4}-", f.stem,
                       re.IGNORECASE)
        data = _read_umafall_file(f, position)
        if data is None or len(data) < UMAFALL_FS:
            continue
        trials.append({
            "data": downsample(data, UMAFALL_FS, FS_TARGET),
            "label": 1 if is_fall else 0

In [13]:
print("Modul geladen aus:", fl.__file__)
print()
!grep -n "def load_umafall" /content/fall_lib.py
!grep -n "unknown" /content/fall_lib.py
!find / -name "fall_lib.py" -not -path "*/proc/*" 2>/dev/null

Modul geladen aus: /content/fall_lib.py

433:def load_umafall(root, position="ANKLE", verbose=True):
454:            "activity": ma.group(1) if ma else "unknown",
/content/fall_lib.py
